# Utforsk vêr- og vassdata

Bruk **proxy-delen** til å sjå kva Respons får frå PythonAnywhere. Bruk **leverandørdelane** til å spørje kjeldene direkte frå notebook-kjernen. Kvar del har eigne eksempel du kan endre og køyre om att.

1. Legg notebooken i prosjektroten på PythonAnywhere, ved sida av frost_sync-mappa. Du kan òg setje PROJECT_DIR nedanfor.
2. Vel ein Python 3.11+-kjerne med prosjektavhengnadene. Frå Bash i miljøet til kjernen kan du køyre: python -m pip install -r requirements.txt pandas matplotlib. Pandas og matplotlib er valfrie og gir tabellvising og plott.
3. Køyr oppsettet og vel deretter cellene for det du vil utforske. Set PROXY_BASE_URL til nettadressa di med ruteprefikset, vanlegvis /weather.

Koden brukar miljøvariablar og prosjektet si .env-fil. Nøklane skal liggje der. Ei .env-fil for webappen som er lagra ein annan stad, kan peikast ut med ENV_FILE. Jupyter-grensesnittet krev betalt konto ifølgje [PythonAnywhere](https://help.pythonanywhere.com/pages/Education/). Sjå [oppsett av eigen notebook-kjerne](https://help.pythonanywhere.com/pages/IPythonNotebookVirtualenvs/).

In [ ]:
import json
import os
import sys
from dataclasses import asdict
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import quote, urlsplit

import requests

# Endre stiane dersom notebooken ligg utanfor repoet eller webappen brukar ei anna .env-fil.
PROJECT_DIR = Path.cwd()
ENV_FILE = Path(os.environ.get("FROST_ENV_FILE") or PROJECT_DIR / ".env")
if not (PROJECT_DIR / "frost_sync" / "config.py").is_file():
    raise RuntimeError("Set PROJECT_DIR til prosjektroten, til dømes Path.home() / 'frost_codex'.")

sys.path.insert(0, str(PROJECT_DIR.resolve()))
os.environ["FROST_ENV_FILE"] = str(ENV_FILE.resolve())

from frost_sync.config import load_settings
from frost_sync.frost_api import FrostClient
from frost_sync.nve_hydapi import NveHydApiClient
from frost_sync.nve_veps import NveVepsClient
from frost_sync.skred_api import SkredClient
from frost_sync.snower_api import SnowerClient
from frost_sync.tilsig_api import TilsigClient
from frost_sync.wyssen_api import WyssenClient

settings = load_settings()
PROXY_BASE_URL = os.getenv("FROST_NOTEBOOK_PROXY_URL", "").rstrip("/")
# Døme: PROXY_BASE_URL = "https://dittnamn.pythonanywhere.com/weather"

try:
    import pandas as pd
except ImportError:
    pd = None
try:
    from IPython.display import display
except ImportError:
    display = print

# Tid i UTC. Køyr cella på nytt for å flytte standardvindauget fram.
TO_UTC = datetime.now(timezone.utc).replace(microsecond=0)
FROM_UTC = TO_UTC - timedelta(days=2)
print("Tidsvindauge:", FROM_UTC.isoformat(), "til", TO_UTC.isoformat())

In [ ]:
def require_key(value, name):
    if not value:
        raise RuntimeError(f"Set {name} i .env eller i miljøet og køyr oppsettet på nytt.")
    return value


def query(label, call, *args, **kwargs):
    """Vis berre feiltype og HTTP-status. Leverandørfeil kan innehalde heile URL-en."""
    try:
        return call(*args, **kwargs)
    except Exception as exc:
        response = getattr(exc, "response", None)
        status = f", HTTP {response.status_code}" if response is not None else ""
        raise RuntimeError(f"{label}: {type(exc).__name__}{status}. Sjekk nøkkel, adresse og tilgang.") from None


def show(rows, columns=None, limit=15):
    rows = list(rows)
    print(f"{len(rows)} rader")
    preview = rows[:limit]
    if columns:
        preview = [{key: row.get(key) for key in columns} for row in preview]
    display(pd.DataFrame(preview) if pd is not None else preview)


def points(rows):
    """Gjer leverandørklientane sine målegrupper om til éi rad per måling."""
    return [
        {
            "station": group.get("sourceId"),
            "time": group.get("referenceTime"),
            "parameter": item.get("elementId"),
            "value": item.get("value"),
            "unit": item.get("unit"),
            "quality": item.get("qualityCode"),
        }
        for group in rows
        for item in group.get("observations", [])
    ]


def iso(value):
    return value.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def proxy_get(path, params=None, *, protected=True):
    base = PROXY_BASE_URL.rstrip("/")
    address = urlsplit(base)
    if address.scheme != "https" or not address.hostname or address.username or address.password or address.query or address.fragment:
        raise ValueError("Set PROXY_BASE_URL til HTTPS-adressa for din eigen proxy, med eventuelt /weather-prefiks.")
    headers = {"Accept": "application/json"}
    if protected:
        headers["X-API-Key"] = require_key(settings.proxy_api_key, "PROXY_API_KEY")
    response = query("PythonAnywhere", requests.get, base + path, params=params,
                     headers=headers, timeout=settings.request_timeout_seconds, allow_redirects=False)
    if not 200 <= response.status_code < 300:
        hint = " Sjekk PROXY_API_KEY." if response.status_code == 401 else " Sjekk adresse, prefiks og parameterar."
        raise RuntimeError(f"PythonAnywhere svarte HTTP {response.status_code}." + hint)
    return query("PythonAnywhere JSON", response.json)


def graph_points(payload):
    """Støttar både eitt-stasjons- og fleirstasjonsresponsen."""
    stations = payload.get("stations", [payload])
    return [
        {"station": group["station"]["source_id"], "time": point["time"],
         "parameter": name, "value": point["value"], "unit": series.get("unit"),
         "quality": point.get("quality_code")}
        for group in stations
        for name, series in group["series"].items()
        for point in series["data"]
    ]

## PythonAnywhere · kartdata og stasjonar

Latest-laga er opne. Vel variant, og filtrer tabellen på leverandør eller stasjonsnamn. Standardvalet viser Snower-stasjonane som er synlege i proxyen.

In [ ]:
# Variant: latest, latest.compact, latest.fme eller latest.7d
variant = "latest"
provider_filter = "snower"  # Bruk "" for alle leverandørar.
name_filter = ""
latest = proxy_get(f"/api/stations/{variant}.geojson", protected=False)
stations = [feature["properties"] for feature in latest["features"]]
matches = [
    station for station in stations
    if (not provider_filter or station.get("provider") == provider_filter)
    and name_filter.casefold() in str(station.get("name") or "").casefold()
]
show(matches, ["source_id", "name", "provider", "observed_at", "snow_depth",
               "air_temperature", "surface_temperature", "surface_temperature_observed_at",
               "has_surface_temperature", "delta_t", "discharge", "discharge_hbv", "discharge_hbv_forecast_time"])

## PythonAnywhere · lagra historikk og tidsseriar

Stasjonsobservasjonane kjem frå databasen. Tidsserierutene spør leverandøren gjennom proxyen. Begge krev PROXY_API_KEY. Parameterkatalogen viser kva logiske feltnamn grafkalla kan bruke.

In [ ]:
parameter_catalog = proxy_get("/api/parameters")
show(parameter_catalog, ["id", "label", "unit"], limit=40)

In [ ]:
source_id = "12.215.0"  # Storeskar. Byt til ein ID frå kartdataa.
station_path = "/api/stations/" + quote(source_id, safe="")
period = {"from": iso(FROM_UTC), "to": iso(TO_UTC)}
detail = proxy_get(station_path)
print(detail.get("name"), detail.get("provider"))
stored = proxy_get(station_path + "/observations", period)
show(stored["observations"], ["reference_time", "element_id", "value", "unit", "quality_code"])

In [ ]:
# Fleire stasjonar kan førast opp i lista.
graph_stations = ["12.215.0"]
graph_parameters = ["discharge", "discharge_hbv"]
graph = proxy_get("/api/timeseries", {
    "stations": ",".join(graph_stations),
    "parameters": ",".join(graph_parameters),
    "from": iso(FROM_UTC),
    "to": iso(TO_UTC + timedelta(days=1)),  # Inkluder framtidige HBV-punkt.
})
print("Manglande stasjonar:", graph.get("missing", []))
print("Stasjonar med kjeldefeil:", [item.get("source_id") for item in graph.get("errors", [])])
graph_rows = graph_points(graph)
show(graph_rows)

## Sjekk overflatetemperatur for SN15951

Latest-tabellen over viser berre kolonnane i `show(...)` og kan vere filtrert på leverandør (standard: Snower). Manglande kolonne er derfor ikkje det same som manglande data. `/api/stations` åleine er ikkje eit endepunkt; bruk `/api/stations/latest.geojson`, `/api/stations/SN15951` eller tidsserieruta nedanfor.

Denne cella brukar same proxy som VertiGIS, ber eksplisitt om begge temperaturane og viser heile temperaturresponsen. Dersom ho fungerer, sjekk at Web Request i VertiGIS òg har `surface_temperature` i `parameters`, og at plottet les den nye responsen. Ved HTTP 401 må notebooken bruke same `PROXY_API_KEY` som webappen. Dersom serveren ikkje kjenner parameteren, oppdater koden og last webappen på nytt.


In [ ]:
surface_station = "SN15951"
surface_path = "/api/stations/" + quote(surface_station, safe="")
surface_graph = proxy_get(surface_path + "/timeseries", {
    "parameters": "air_temperature,surface_temperature",
    "from": iso(FROM_UTC),
    "to": iso(TO_UTC),
})
for parameter, item in surface_graph["series"].items():
    print(parameter, item.get("unit"), "punkt:", len(item.get("data", [])))
surface_rows = points([
    {"sourceId": surface_station, "referenceTime": point["time"],
     "observations": [{"elementId": parameter, "value": point["value"], "unit": item.get("unit")}]}
    for parameter, item in surface_graph["series"].items()
    for point in item.get("data", [])
])
show(surface_rows, ["station", "time", "parameter", "value", "unit"], limit=30)


## Direkte frå MET Frost, NVE HydAPI og VEPS

Frost bruker element-ID-ar frå MET. HydAPI-klienten finn seriar for vald stasjon, medan VEPS returnerer HBV som modellert døgnmiddel. Kvar celle kontaktar berre den oppgitte kjelda.

In [ ]:
frost_id = "SN18700"
frost_elements = ["air_temperature", "sum(precipitation_amount PT1H)"]
frost = FrostClient(settings.frost_base_url, require_key(settings.frost_client_id, "FROST_CLIENT_ID"),
                    timeout_seconds=settings.request_timeout_seconds,
                    acceptable_quality_codes=settings.acceptable_quality_codes)
frost_rows = points(query("Frost", frost.fetch_observations_range,
                          frost_id, frost_elements, FROM_UTC, TO_UTC))
show(frost_rows)

In [ ]:
nve_id = "12.215.0"
nve_parameter = "discharge"
hydapi = NveHydApiClient(settings.nve_hydapi_base_url,
                       require_key(settings.nve_hydapi_key, "NVE_HYDAPI_KEY"),
                       timeout_seconds=settings.request_timeout_seconds,
                       veps_base_url=settings.nve_veps_base_url)
specs = query("HydAPI seriar", hydapi.fetch_series_specs_for_station, nve_id)
show(asdict(spec) for spec in specs)
chosen = [spec for spec in specs if spec.logical_element_id == nve_parameter]
hydapi_rows = points(query("HydAPI målingar", hydapi.fetch_observations_range,
                           chosen, FROM_UTC, TO_UTC))
show(hydapi_rows)

In [ ]:
veps_id = "12.215.0"
veps = NveVepsClient(settings.nve_veps_base_url, timeout_seconds=settings.request_timeout_seconds)
veps_rows = points(query("VEPS", veps.fetch_observations_range,
                         [veps_id], FROM_UTC, TO_UTC + timedelta(days=1)))
show(veps_rows)

## Direkte frå Snower

Den første cella listar monitorar i leverandørkontoen med eitt kall. Den neste sjekkar aktivstatus for éin vald monitor. Du kan samanlikne denne statusen med kartlaget i proxy-delen. Proxyen oppdaterer aktivlista når sync-metadata køyrer.

In [ ]:
snower = SnowerClient(settings.snower_base_url,
                      require_key(settings.snower_api_key, "SNOWER_API_KEY"),
                      domain_id=settings.snower_domain_id or "frost-sync",
                      timeout_seconds=settings.request_timeout_seconds)
snower_monitors = query("Snower monitorliste", snower._post, "/monitors_list", {"format": "extended"})
show(snower_monitors, ["permanent_id", "name"])

In [ ]:
# Bruk permanent_id frå Snower eller ein source_id med SNWR-prefiks frå kartlaget.
snower_id = ""
if not snower_id:
    print("Set snower_id til monitoren du vil sjekke.")
else:
    permanent_id = snower_id.removeprefix("SNWR-")
    monitor_request = {"monitor": permanent_id}
    status = query("Snower aktivstatus", snower._post, "/monitor_active_status", monitor_request)
    print("active_status:", status.get("active_status"))
    context = json.dumps({"permanent_id": permanent_id})
    snower_rows = points(query("Snower historikk", snower.fetch_observations_range,
                               context, FROM_UTC, TO_UTC, ["snow_depth", "air_temperature"]))
    show(snower_rows)

## Direkte frå Tilsig, Wyssen og Skred AS

Cellene viser stasjonar og hentar historikk for éin av dei. Set den aktuelle ID-en for å velje stasjon. Med tom ID brukar cella første stasjon i lista. Verdiane er normaliserte av dei same kjeldeklientane som proxyen brukar.

In [ ]:
def monitor_points(label, client, source_id, parameters):
    monitors = query(label + " stasjonar", client.fetch_stations)
    show((asdict(item) for item in monitors), ["source_id", "name", "longitude", "latitude"])
    if not monitors:
        return []
    selected = next((item for item in monitors if item.source_id == source_id), None) if source_id else monitors[0]
    if selected is None:
        raise ValueError(f"Vel ein source_id frå stasjonslista for {label}.")
    print("Vald stasjon:", selected.source_id)
    return points(query(label + " historikk", client.fetch_observations_range,
                        selected.provider_context, FROM_UTC, TO_UTC, parameters))

In [ ]:
tilsig_id = ""
tilsig = TilsigClient(base_url=settings.tilsig_base_url,
                      api_key=require_key(settings.tilsig_api_key, "TILSIG_API_KEY"),
                      timeout_seconds=settings.request_timeout_seconds)
tilsig_rows = monitor_points("Tilsig", tilsig, tilsig_id, ["snow_depth"])
show(tilsig_rows)

In [ ]:
wyssen_id = ""
wyssen = WyssenClient(api_key=require_key(settings.wyssen_api_key, "WYSSEN_API_KEY"),
                      timeout_seconds=settings.request_timeout_seconds)
wyssen_rows = monitor_points("Wyssen", wyssen, wyssen_id,
                             ["air_temperature", "wind_speed", "wind_from_direction"])
show(wyssen_rows)

In [ ]:
skred_id = "SKRED-cs001"
skred = SkredClient(base_url=settings.skred_base_url,
                    api_token=require_key(settings.skred_api_token, "SKRED_API_TOKEN"),
                    timeout_seconds=settings.request_timeout_seconds)
skred_rows = monitor_points("Skred AS", skred, skred_id,
                            ["air_temperature", "wind_speed", "snow_drift_low", "snow_drift_high"])
show(skred_rows)

## Samanlikn vassføring i ein graf

Køyr først cellene for proxy-tidsseriar, HydAPI og VEPS. Plottet brukar berre vassføringsseriar med same eining. HBV er modellert vassføring og har anna tidsoppløysing enn observerte punkt.

In [ ]:
import matplotlib.pyplot as plt

if pd is None:
    raise RuntimeError("Installer pandas i notebook-kjernen for å bruke grafcella.")
frames = [
    pd.DataFrame(rows).assign(kjelde=label)
    for label, rows in [
        ("Proxy", globals().get("graph_rows", [])),
        ("HydAPI direkte", globals().get("hydapi_rows", [])),
        ("VEPS direkte", globals().get("veps_rows", [])),
    ]
    if rows
]
if not frames:
    print("Køyr ei av vassføringscellene først.")
else:
    data = pd.concat(frames, ignore_index=True)
    data = data[data["parameter"].isin(["discharge", "discharge_hbv"])].copy()
    data["time"] = data["time"].map(lambda value: pd.to_datetime(value, utc=True, errors="coerce"))
    data["value"] = pd.to_numeric(data["value"], errors="coerce")
    data["unit"] = data["unit"].replace({"m³/s": "m3/s", "m^3/s": "m3/s"})
    data = data[data["unit"].eq("m3/s")].dropna(subset=["time", "value"])
    if data.empty:
        print("Ingen vassføringspunkt i m³/s i dei valde svara.")
    else:
        fig, ax = plt.subplots(figsize=(11, 4))
        for (source, station, parameter), group in data.groupby(["kjelde", "station", "parameter"]):
            group = group.sort_values("time")
            ax.plot(group["time"], group["value"], label=f"{source} · {station} · {parameter}",
                    marker="o" if parameter == "discharge_hbv" else None)
        ax.set(xlabel="Tid (UTC)", ylabel="Vassføring (m³/s)")
        ax.grid(alpha=0.25)
        ax.legend()
        fig.autofmt_xdate()
        plt.show()

## Valfritt · les lagra målingar direkte frå databasen

Denne cella brukar DATABASE_URL frå prosjektoppsettet og gjer ei avgrensa SELECT-spørring. Ho er meint for deg som har tilgang til databasen på PythonAnywhere. Ei direkte databasespørring kan også vise lagra målingar for stasjonar som proxyen har skjult.

In [ ]:
RUN_DATABASE_QUERY = False  # Set True for å spørje databasen.
database_station = "12.215.0"

if RUN_DATABASE_QUERY:
    from sqlalchemy import create_engine, select
    from sqlalchemy.engine import make_url
    from frost_sync.models import Observation, Station

    url = make_url(settings.database_url)
    if url.get_backend_name() == "sqlite":
        database_file = (PROJECT_DIR / (url.database or "")).resolve()
        if not database_file.is_file():
            raise RuntimeError("Fann ikkje SQLite-fila. Sjekk PROJECT_DIR og DATABASE_URL.")
        url = url.set(database=str(database_file))
    engine = query("Databaseoppsett", create_engine, url)
    statement = (
        select(Station.source_id, Observation.reference_time, Observation.element_id,
               Observation.value, Observation.unit)
        .join(Station, Station.id == Observation.station_id)
        .where(Station.source_id == database_station)
        .order_by(Observation.reference_time.desc())
        .limit(100)
    )

    def read_observations():
        with engine.connect() as connection:
            return list(connection.execute(statement).mappings())

    try:
        show(query("Lagra observasjonar", read_observations))
    finally:
        engine.dispose()